# Homework 5.1 - Step 2: Syllable Tokenization

This notebook performs the syllable-tokenization step for the two cleaned corpora created in Step 1.

The source corpus is Marcus Aurelius' *Meditations*.

The independent English reference corpus is Plato's *The Republic*, translated by Benjamin Jowett.

The goal is to convert normalized English words into reproducible written-syllable segments, construct integer token vocabularies, compute syllable frequencies, verify exact reconstruction, and preserve every source syllable as a distinct token.

## Modeling idea

Let the cleaned source text be a sequence of normalized words. Each word is split into written syllable segments.

For a normalized word,

$$
w = s_1 s_2 \cdots s_m
$$

where each $$s_j$$ is a written syllable segment and concatenating the segments must reproduce the original normalized word exactly.

For the full source corpus, the syllable sequence is

$$
x_1, x_2, \ldots, x_L.
$$

The source vocabulary is

$$
T = \{t_1, t_2, \ldots, t_K\},
$$

where each distinct written syllable receives its own integer ID.

Here:

- $$L$$ is the total number of syllable occurrences in *Meditations*.
- $$K$$ is the number of distinct syllable types in *Meditations*.

The same syllabification rule is applied to *The Republic*. This gives an independent English reference distribution over written syllables.

## Syllabification convention

This notebook uses `Pyphen` with the `en_US` hyphenation patterns as a reproducible written-syllable approximation.

This is a written segmentation rule, not a claim about exact phonetic syllables. Hyphenation boundaries can differ from pronunciation boundaries, especially for names, archaic spellings, contractions, and unusual words.

The same `Pyphen` configuration is applied to both corpora.

A deterministic fallback is also included. If the primary method ever returns an invalid segmentation or fails to reconstruct the original word, a vowel-based written segmentation heuristic is used. The fallback preserves every character and therefore still permits an exact reconstruction check.

Because `Pyphen` is pattern-based rather than a closed pronunciation dictionary, an unchanged word does not necessarily mean the word is absent from the resource. The notebook therefore reports both:

- the number of words for which the fallback was actually required,
- examples of longer words that the primary method left unsplit.

These are linguistic/tokenization limitations, not decipherment errors.

## Handling source syllables missing from the reference corpus

No source syllable is replaced by a common unknown token.

The source vocabulary is built first from *Meditations*. Every distinct source syllable receives a permanent source ID.

If a source syllable does not occur in *The Republic*, its reference frequency is simply recorded as zero.

For later use, a union vocabulary is also created. It contains:

1. every source syllable, keeping the same source ID,
2. any additional syllables found only in *The Republic*, assigned new IDs after the source vocabulary.

Therefore two different source syllables can never be merged into one `<UNK>` category merely because they are absent from the reference corpus.

In [1]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import csv
import re

try:
    import pyphen
except ImportError as exc:
    raise ImportError(
        "Pyphen is required for this notebook. "
        "Run `%pip install pyphen` in a notebook cell, restart the kernel if needed, "
        "and then run this notebook again."
    ) from exc

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MEDITATIONS_PATH = DATA_DIR / "meditations_cleaned.txt"
REPUBLIC_PATH = DATA_DIR / "republic_cleaned.txt"

if not MEDITATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MEDITATIONS_PATH}. Run the Step 1 corpus-preparation notebook first."
    )

if not REPUBLIC_PATH.exists():
    raise FileNotFoundError(
        f"Missing {REPUBLIC_PATH}. Run the Step 1 corpus-preparation notebook first."
    )

meditations_text = MEDITATIONS_PATH.read_text(encoding="utf-8")
republic_text = REPUBLIC_PATH.read_text(encoding="utf-8")

print("Pyphen version:", version("pyphen"))
print("Meditations characters:", f"{len(meditations_text):,}")
print("Republic characters:", f"{len(republic_text):,}")

Pyphen version: 0.18.1
Meditations characters: 312,633
Republic characters: 641,395


## Tokenization normalization

The Gutenberg translation contains historical Latin ligatures inside English words, including forms such as `Cæsar`, `Mæcenas`, and `Crœsus`.

Before syllabification, these written forms are normalized consistently in both corpora:

- `æ` becomes `ae`,
- `œ` becomes `oe`,
- common typographic Latin ligatures such as `ﬁ` and `ﬂ` are expanded.

This normalization is frozen before the decipherment experiment. It prevents a word such as `cæsar` from being incorrectly split into the ASCII fragments `c` and `sar`.

The resulting normalized corpus files are saved separately and are the canonical text inputs for Tasks 2 and 3.

In [2]:
import unicodedata
from collections import Counter

TOKENIZATION_CHAR_MAP = str.maketrans({
    "æ": "ae",
    "Æ": "AE",
    "œ": "oe",
    "Œ": "OE",
    "ﬀ": "ff",
    "ﬁ": "fi",
    "ﬂ": "fl",
    "ﬃ": "ffi",
    "ﬄ": "ffl",
})


def normalize_for_tokenization(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(TOKENIZATION_CHAR_MAP)
    return text.lower()


meditations_text = normalize_for_tokenization(meditations_text)
republic_text = normalize_for_tokenization(republic_text)

MEDITATIONS_TOKENIZATION_PATH = DATA_DIR / "meditations_tokenization_text.txt"
REPUBLIC_TOKENIZATION_PATH = DATA_DIR / "republic_tokenization_text.txt"

MEDITATIONS_TOKENIZATION_PATH.write_text(
    meditations_text,
    encoding="utf-8",
)

REPUBLIC_TOKENIZATION_PATH.write_text(
    republic_text,
    encoding="utf-8",
)


def remaining_non_ascii_letters(text):
    return Counter(
        ch for ch in text
        if ch.isalpha() and not ch.isascii()
    )


med_non_ascii = remaining_non_ascii_letters(meditations_text)
rep_non_ascii = remaining_non_ascii_letters(republic_text)

print("Saved canonical tokenization texts:")
print(MEDITATIONS_TOKENIZATION_PATH.resolve())
print(REPUBLIC_TOKENIZATION_PATH.resolve())

print()
print("Remaining non-ASCII alphabetic characters in Meditations:")
print(dict(med_non_ascii.most_common(30)))

print()
print("Remaining non-ASCII alphabetic characters in Republic:")
print(dict(rep_non_ascii.most_common(30)))

Saved canonical tokenization texts:
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\meditations_tokenization_text.txt
C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\republic_tokenization_text.txt

Remaining non-ASCII alphabetic characters in Meditations:
{'ν': 21, 'ο': 17, 'μ': 17, 'ρ': 15, 'ε': 14, 'α': 14, 'ς': 12, 'τ': 11, 'ι': 11, 'σ': 10, 'φ': 9, 'ω': 9, 'ί': 8, 'κ': 7, 'π': 5, 'ό': 5, 'θ': 5, 'έ': 5, 'ἀ': 4, 'δ': 3, 'υ': 3, 'ἔ': 3, 'ὐ': 2, 'λ': 2, 'β': 2, 'ύ': 2, 'ὑ': 2, 'ἰ': 1, 'ή': 1, 'ῦ': 1}

Remaining non-ASCII alphabetic characters in Republic:
{}


## Word extraction

Punctuation remains side information and is not treated as a syllable token.

For this syllable-counting step, a normalized word is taken to be a contiguous alphabetic sequence. Apostrophes, hyphens, quotation marks, commas, periods, and other punctuation therefore remain outside the syllable vocabulary.

The same rule is used for both corpora.

In [3]:
WORD_RE = re.compile(r"[a-z]+")


def extract_words(text):
    return WORD_RE.findall(text.lower())


meditation_words = extract_words(meditations_text)
republic_words = extract_words(republic_text)

print("Meditations word occurrences:", f"{len(meditation_words):,}")
print("Republic word occurrences:", f"{len(republic_words):,}")
print("Meditations distinct words:", f"{len(set(meditation_words)):,}")
print("Republic distinct words:", f"{len(set(republic_words)):,}")

Meditations word occurrences: 57,029
Republic word occurrences: 119,005
Meditations distinct words: 4,324
Republic distinct words: 7,289


## Define the written-syllable tokenizer

In [4]:
HYPHENATOR = pyphen.Pyphen(lang="en_US")
VOWELS = set("aeiouy")


def fallback_written_syllables(word):
    # Deterministic vowel-based fallback.
    # The goal is a stable written segmentation that preserves every character.
    if not word:
        return []

    nuclei = []
    i = 0

    while i < len(word):
        if word[i] in VOWELS:
            start = i

            while i + 1 < len(word) and word[i + 1] in VOWELS:
                i += 1

            end = i + 1
            nuclei.append((start, end))

        i += 1

    # Words with zero or one visible vowel nucleus remain as one segment.
    if len(nuclei) <= 1:
        return [word]

    boundaries = []

    for j in range(len(nuclei) - 1):
        current_end = nuclei[j][1]
        next_start = nuclei[j + 1][0]
        cluster_length = next_start - current_end

        if cluster_length <= 1:
            boundary = current_end
        else:
            boundary = current_end + 1

        boundaries.append(boundary)

    pieces = []
    start = 0

    for boundary in boundaries:
        pieces.append(word[start:boundary])
        start = boundary

    pieces.append(word[start:])
    pieces = [piece for piece in pieces if piece]

    if "".join(pieces) != word:
        raise RuntimeError(f"Fallback reconstruction failed for {word!r}")

    return pieces


def syllabify_word(word):
    try:
        inserted = HYPHENATOR.inserted(word)
        segments = [part for part in inserted.split("-") if part]

        valid = bool(segments) and "".join(segments) == word

        if valid:
            return segments, False

    except Exception:
        pass

    segments = fallback_written_syllables(word)

    if "".join(segments) != word:
        raise RuntimeError(f"Reconstruction failed for {word!r}")

    return segments, True

## Tokenize both corpora with identical rules

In [5]:
def tokenize_words(words):
    syllable_sequence = []
    word_segmentations = {}
    fallback_counter = Counter()
    unsplit_counter = Counter()

    for word in words:
        if word in word_segmentations:
            segments, used_fallback = word_segmentations[word]
        else:
            segments, used_fallback = syllabify_word(word)
            word_segmentations[word] = (segments, used_fallback)

        if "".join(segments) != word:
            raise AssertionError(f"Exact reconstruction failed for {word!r}")

        syllable_sequence.extend(segments)

        if used_fallback:
            fallback_counter[word] += 1

        if len(segments) == 1 and len(word) >= 7:
            unsplit_counter[word] += 1

    return {
        "syllable_sequence": syllable_sequence,
        "word_segmentations": word_segmentations,
        "fallback_counter": fallback_counter,
        "unsplit_counter": unsplit_counter,
    }


med_result = tokenize_words(meditation_words)
rep_result = tokenize_words(republic_words)

med_syllables = med_result["syllable_sequence"]
rep_syllables = rep_result["syllable_sequence"]

print("Tokenization complete.")
print("Meditations syllable occurrences:", f"{len(med_syllables):,}")
print("Republic syllable occurrences:", f"{len(rep_syllables):,}")

Tokenization complete.
Meditations syllable occurrences: 77,260
Republic syllable occurrences: 157,157


## Verify exact reconstruction

In [6]:
def verify_all_reconstructions(word_segmentations):
    failures = []

    for word, (segments, used_fallback) in word_segmentations.items():
        reconstructed = "".join(segments)

        if reconstructed != word:
            failures.append((word, segments, reconstructed))

    return failures


med_failures = verify_all_reconstructions(
    med_result["word_segmentations"]
)

rep_failures = verify_all_reconstructions(
    rep_result["word_segmentations"]
)

print("Meditations reconstruction failures:", len(med_failures))
print("Republic reconstruction failures:", len(rep_failures))

assert len(med_failures) == 0
assert len(rep_failures) == 0

print("Exact reconstruction verified for every distinct normalized word.")

Meditations reconstruction failures: 0
Republic reconstruction failures: 0
Exact reconstruction verified for every distinct normalized word.


## Construct the source vocabulary

In [7]:
med_frequency = Counter(med_syllables)
rep_frequency = Counter(rep_syllables)

source_syllables = sorted(med_frequency)

source_id = {
    syllable: idx
    for idx, syllable in enumerate(source_syllables)
}

L = len(med_syllables)
K = len(source_syllables)

print("Source corpus: Meditations")
print("L =", f"{L:,}", "total syllable occurrences")
print("K =", f"{K:,}", "distinct source syllable types")

Source corpus: Meditations
L = 77,260 total syllable occurrences
K = 3,052 distinct source syllable types


The source token sequence is obtained by replacing each source syllable with its permanent integer ID.

$$
x_1, x_2, \ldots, x_L
\quad\longrightarrow\quad
\text{ID}(x_1), \text{ID}(x_2), \ldots, \text{ID}(x_L).
$$

In [8]:
source_token_ids = [
    source_id[syllable]
    for syllable in med_syllables
]

assert len(source_token_ids) == L

print("First 40 source syllable tokens:")
print(med_syllables[:40])

print()
print("First 40 source token IDs:")
print(source_token_ids[:40])

First 40 source syllable tokens:
['of', 'my', 'grand', 'fa', 'ther', 'verus', 'i', 'have', 'learned', 'to', 'be', 'gen', 'tle', 'and', 'meek', 'and', 'to', 're', 'frain', 'from', 'all', 'anger', 'and', 'pas', 'sion', 'from', 'the', 'fame', 'and', 'mem', 'o', 'ry', 'of', 'him', 'that', 'be', 'got', 'me', 'i', 'have']

First 40 source token IDs:
[1697, 1590, 1053, 791, 2630, 2875, 1217, 1133, 1354, 2712, 159, 1002, 2707, 76, 1511, 76, 2712, 2040, 953, 966, 64, 77, 76, 1763, 2310, 966, 2619, 811, 76, 1517, 1683, 2140, 1697, 1173, 2618, 159, 1047, 1503, 1217, 1133]


## Construct the reference and union vocabularies

In [9]:
reference_syllables = sorted(rep_frequency)
reference_K = len(reference_syllables)
reference_L = len(rep_syllables)

source_set = set(source_syllables)
reference_set = set(reference_syllables)

shared_syllables = source_set & reference_set
source_only_syllables = source_set - reference_set
reference_only_syllables = reference_set - source_set

union_id = dict(source_id)

next_id = K

for syllable in sorted(reference_only_syllables):
    union_id[syllable] = next_id
    next_id += 1

reference_token_ids = [
    union_id[syllable]
    for syllable in rep_syllables
]

print("Reference corpus: The Republic")
print("Reference syllable occurrences:", f"{reference_L:,}")
print("Reference distinct syllables:", f"{reference_K:,}")

print()
print("Shared syllables:", f"{len(shared_syllables):,}")
print("Source-only syllables:", f"{len(source_only_syllables):,}")
print("Reference-only syllables:", f"{len(reference_only_syllables):,}")
print("Union vocabulary size:", f"{len(union_id):,}")

Reference corpus: The Republic
Reference syllable occurrences: 157,157
Reference distinct syllables: 4,410

Shared syllables: 2,240
Source-only syllables: 812
Reference-only syllables: 2,170
Union vocabulary size: 5,222


## Why there is no unknown-token merging

Suppose two source syllables $$a$$ and $$b$$ both fail to appear in *The Republic*.

They remain two different source vocabulary entries:

$$
\text{ID}(a) \neq \text{ID}(b).
$$

Their reference frequencies are simply

$$
f_{\text{Republic}}(a)=0,
\qquad
f_{\text{Republic}}(b)=0.
$$

No common `<UNK>` token is introduced for missing source syllables.

## Show representative word-to-syllable examples

In [10]:
def choose_examples(word_segmentations, preferred=None, n=15):
    preferred = preferred or []
    examples = []
    used = set()

    for word in preferred:
        if word in word_segmentations and word not in used:
            segments, fallback = word_segmentations[word]
            examples.append((word, segments, fallback))
            used.add(word)

    candidates = sorted(
        word_segmentations,
        key=lambda w: (-len(w), w),
    )

    for word in candidates:
        if word in used:
            continue

        segments, fallback = word_segmentations[word]

        if len(segments) >= 2:
            examples.append((word, segments, fallback))
            used.add(word)

        if len(examples) >= n:
            break

    return examples[:n]


preferred_words = [
    "meditations",
    "philosophy",
    "emperor",
    "justice",
    "nature",
    "reason",
    "virtue",
    "republic",
    "socrates",
    "knowledge",
]

examples = choose_examples(
    med_result["word_segmentations"],
    preferred=preferred_words,
    n=15,
)

print(f"{'Word':<24} Segmentation")
print("-" * 70)

for word, segments, used_fallback in examples:
    marker = " [fallback]" if used_fallback else ""
    print(f"{word:<24} {' | '.join(segments)}{marker}")

Word                     Segmentation
----------------------------------------------------------------------
philosophy               phi | los | o | phy
justice                  jus | tice
nature                   na | ture
reason                   rea | son
virtue                   virtue
socrates                 socrates
knowledge                knowl | edge
unconscionableness       un | con | scionable | ness
incomprehensible         in | com | pre | hen | si | ble
prestidigitators         pres | tidig | i | ta | tors
uncharitableness         un | char | i | ta | ble | ness
unpassionateness         un | pas | sion | ate | ness
contentiousness          con | tentious | ness
excrementitious          ex | cre | men | ti | tious
notwithstanding          not | with | stand | ing


## Inspect fallback usage and unsplit long words

In [11]:
med_fallback_occurrences = sum(
    med_result["fallback_counter"].values()
)

rep_fallback_occurrences = sum(
    rep_result["fallback_counter"].values()
)

print("Fallback occurrences")
print("--------------------")
print("Meditations:", med_fallback_occurrences)
print("Republic:", rep_fallback_occurrences)

print()
print("Long Meditations words left unsplit by the primary rule:")
for word, count in med_result["unsplit_counter"].most_common(20):
    print(f"{word:<24} {count}")

print()
print("Long Republic words left unsplit by the primary rule:")
for word, count in rep_result["unsplit_counter"].most_common(20):
    print(f"{word:<24} {count}")

Fallback occurrences
--------------------
Meditations: 0
Republic: 0

Long Meditations words left unsplit by the primary rule:
present                  68
whether                  62
against                  57
through                  43
thoughts                 25
brought                  21
thought                  15
friends                  14
pleased                  13
socrates                 12
wouldst                  12
learned                  11
straight                 11
grieved                  11
changes                  11
changed                  10
chances                  9
shouldest                8
presents                 7
greater                  6

Long Republic words left unsplit by the primary rule:
replied                  254
whether                  177
guardians                80
socrates                 69
against                  68
greater                  62
thought                  49
friends                  48
present                  45
stronger

## Most frequent syllables

In [12]:
print("Meditations: 30 most frequent syllables")
print("-" * 45)

for syllable, count in med_frequency.most_common(30):
    print(f"{syllable:<15} {count:>8,}")

print()
print("Republic: 30 most frequent syllables")
print("-" * 45)

for syllable, count in rep_frequency.most_common(30):
    print(f"{syllable:<15} {count:>8,}")

Meditations: 30 most frequent syllables
---------------------------------------------
and                2,863
to                 2,243
of                 1,979
the                1,861
that               1,820
in                 1,333
er                 1,332
is                 1,272
it                 1,213
be                 1,211
a                    905
so                   866
as                   838
thou                 811
ly                   806
or                   795
for                  755
ing                  658
con                  657
not                  646
un                   641
all                  611
i                    568
thy                  553
what                 523
with                 522
things               520
tion                 516
self                 510
which                510

Republic: 30 most frequent syllables
---------------------------------------------
the                7,103
and                5,755
of                 4,750
to   

## Save the source vocabulary

In [13]:
source_vocab_path = DATA_DIR / "source_syllable_vocabulary.csv"

with source_vocab_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "source_id",
        "syllable",
        "meditations_frequency",
        "republic_frequency",
        "present_in_republic",
    ])

    for syllable in source_syllables:
        writer.writerow([
            source_id[syllable],
            syllable,
            med_frequency[syllable],
            rep_frequency.get(syllable, 0),
            syllable in reference_set,
        ])

print("Saved:", source_vocab_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\source_syllable_vocabulary.csv


## Save the union vocabulary

In [14]:
union_vocab_path = DATA_DIR / "union_syllable_vocabulary.csv"

items_by_id = sorted(
    union_id.items(),
    key=lambda item: item[1],
)

with union_vocab_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "union_id",
        "syllable",
        "in_meditations",
        "in_republic",
        "meditations_frequency",
        "republic_frequency",
    ])

    for syllable, token_id in items_by_id:
        writer.writerow([
            token_id,
            syllable,
            syllable in source_set,
            syllable in reference_set,
            med_frequency.get(syllable, 0),
            rep_frequency.get(syllable, 0),
        ])

print("Saved:", union_vocab_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\union_syllable_vocabulary.csv


## Save the syllable token sequences

In [15]:
source_sequence_path = DATA_DIR / "meditations_syllable_token_sequence.csv"

with source_sequence_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["position", "source_id", "syllable"])

    for position, syllable in enumerate(med_syllables):
        writer.writerow([
            position,
            source_id[syllable],
            syllable,
        ])


reference_sequence_path = DATA_DIR / "republic_syllable_token_sequence.csv"

with reference_sequence_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["position", "union_id", "syllable"])

    for position, syllable in enumerate(rep_syllables):
        writer.writerow([
            position,
            union_id[syllable],
            syllable,
        ])


print("Saved:", source_sequence_path.resolve())
print("Saved:", reference_sequence_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\meditations_syllable_token_sequence.csv
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\republic_syllable_token_sequence.csv


## Save word-to-syllable examples

In [16]:
examples_path = DATA_DIR / "syllable_examples.csv"

with examples_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "word",
        "segments",
        "reconstructed_word",
        "fallback_used",
    ])

    for word, segments, used_fallback in examples:
        writer.writerow([
            word,
            " | ".join(segments),
            "".join(segments),
            used_fallback,
        ])

print("Saved:", examples_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\syllable_examples.csv


## Save Task 2 summary

In [17]:
summary_path = DATA_DIR / "task2_syllable_summary.csv"

summary_rows = [
    ["meditations_total_syllables_L", L],
    ["meditations_vocabulary_size_K", K],
    ["republic_total_syllables", reference_L],
    ["republic_vocabulary_size", reference_K],
    ["shared_syllable_types", len(shared_syllables)],
    ["source_only_syllable_types", len(source_only_syllables)],
    ["reference_only_syllable_types", len(reference_only_syllables)],
    ["union_vocabulary_size", len(union_id)],
    ["meditations_fallback_occurrences", med_fallback_occurrences],
    ["republic_fallback_occurrences", rep_fallback_occurrences],
    ["meditations_reconstruction_failures", len(med_failures)],
    ["republic_reconstruction_failures", len(rep_failures)],
]

with summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["quantity", "value"])
    writer.writerows(summary_rows)

print("Saved:", summary_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\task2_syllable_summary.csv


## Task 2 outputs

After the notebook runs successfully, the `data/` folder contains:

- `meditations_tokenization_text.txt`
- `republic_tokenization_text.txt`
- `source_syllable_vocabulary.csv`
- `union_syllable_vocabulary.csv`
- `meditations_syllable_token_sequence.csv`
- `republic_syllable_token_sequence.csv`
- `syllable_examples.csv`
- `task2_syllable_summary.csv`

The two `*_tokenization_text.txt` files contain the frozen text normalization used by Tasks 2 and 3.

The source vocabulary remains fixed after this step.

The hidden glyph map in Step 3 will be constructed from the distinct source syllables in `source_syllable_vocabulary.csv`.